# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE, ADASYN


# Load Dataset

In [2]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

# Data Preprocessing

In [3]:
# Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)


# Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])


# Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})


# One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)


# Split Features & Target
X = df.drop('y', axis=1)
y = df['y']


# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling

In [4]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Sampling Techniques

In [5]:
smote = SMOTE(random_state=42)
adasyn = ADASYN(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train, y_train)

# Models

In [6]:
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "MLP": MLPClassifier(hidden_layer_sizes=(64,), max_iter=200, early_stopping=True, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=50, learning_rate=0.1, random_state=42, n_jobs=-1)
}

# Evaluation Metrics

In [7]:
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

# Cross Validation Setup

In [8]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

#For storing

In [9]:
results= []

# Function to Evaluate Models

In [10]:
def evaluate_models(X, y, label):
    print(f"\n===== {label} =====")

    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=skf, scoring=scoring)


        metrics={
            "Label":f"{label}",
            "Model": name,
            "Accuracy":  f"{np.mean(scores['test_accuracy']):.4f}",
            "Precision": f"{np.mean(scores['test_precision']):.4f}",
            "Recall":    f"{np.mean(scores['test_recall']):.4f}",
            "F1 Score":  f"{np.mean(scores['test_f1']):.4f}",
            "ROC-AUC":   f"{np.mean(scores['test_roc_auc']):.4f}"
        }
        results.append(metrics)

        print(f"\nModel: {name}")
        print(f"Accuracy:  {np.mean(scores['test_accuracy']):.4f}")
        print(f"Precision: {np.mean(scores['test_precision']):.4f}")
        print(f"Recall:    {np.mean(scores['test_recall']):.4f}")
        print(f"F1 Score:  {np.mean(scores['test_f1']):.4f}")
        print(f"ROC-AUC:   {np.mean(scores['test_roc_auc']):.4f}")
# Run Experiments
# Baseline
evaluate_models(X_train, y_train, "Baseline (No Sampling)")

# SMOTE
evaluate_models(X_train_sm, y_train_sm, "SMOTE")

# ADASYN
evaluate_models(X_train_ad, y_train_ad, "ADASYN")


===== Baseline (No Sampling) =====

Model: Decision Tree
Accuracy:  0.8989
Precision: 0.6277
Recall:    0.2541
F1 Score:  0.3610
ROC-AUC:   0.7750

Model: Random Forest
Accuracy:  0.8993
Precision: 0.6661
Recall:    0.2120
F1 Score:  0.3211
ROC-AUC:   0.7960

Model: MLP
Accuracy:  0.8981
Precision: 0.6247
Recall:    0.2443
F1 Score:  0.3496
ROC-AUC:   0.7868

Model: XGBoost
Accuracy:  0.8999
Precision: 0.6455
Recall:    0.2481
F1 Score:  0.3576
ROC-AUC:   0.7990

===== SMOTE =====

Model: Decision Tree
Accuracy:  0.7663
Precision: 0.8728
Recall:    0.6235
F1 Score:  0.7274
ROC-AUC:   0.8275

Model: Random Forest
Accuracy:  0.8485
Precision: 0.8958
Recall:    0.7886
F1 Score:  0.8388
ROC-AUC:   0.9366

Model: MLP
Accuracy:  0.8598
Precision: 0.8714
Recall:    0.8443
F1 Score:  0.8575
ROC-AUC:   0.9332

Model: XGBoost
Accuracy:  0.9122
Precision: 0.9366
Recall:    0.8843
F1 Score:  0.9097
ROC-AUC:   0.9661

===== ADASYN =====

Model: Decision Tree
Accuracy:  0.7698
Precision: 0.8708
Rec

#Converting into dataframe

In [11]:
results_df = pd.DataFrame(results)
from google.colab import drive

# Save directly to Drive

In [12]:
results_df.to_excel('ML_Project( Using sampling Techniques & Baseline )_result.xlsx', index=False)